In [2]:
import os
model_path = "C:\\Users\\endop\\Downloads\\gesture_recognizer.task"
# 2. Verify the file exists and is a file
if not os.path.isfile(model_path):
    raise FileNotFoundError(
        f"Model file not found at: {os.path.abspath(model_path)}\n"
        "Please download it from: https://storage.googleapis.com/mediapipe-models/"
        "gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task"
    )

print(f"Model file found at: {os.path.abspath(model_path)}")

# 3. Now configure and create the recognizer with the verified path
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.GestureRecognizerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.LIVE_STREAM,
    # ... other options ...
)

Model file found at: C:\Users\endop\Downloads\gesture_recognizer.task


In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# 1. Define the result callback
def print_result(result: vision.GestureRecognizerResult, output_image: mp.Image, timestamp_ms: int):
    if result.gestures:
        # Get the top gesture for the first detected hand
        top_gesture = result.gestures[0][0]
        print(f"Gesture: {top_gesture.category_name} "
              f"(Confidence: {top_gesture.score:.2f})")

    # You can also access hand landmarks from result.hand_landmarks
    # and handedness from result.handedness

# 2. Configure the Gesture Recognizer
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.GestureRecognizerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.LIVE_STREAM,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    result_callback=print_result
)

# 3. Initialize the recognizer and start the webcam loop
with vision.GestureRecognizer.create_from_options(options) as recognizer:
    cap = cv2.VideoCapture(0)  # 0 for default webcam

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        # Convert BGR (OpenCV) to RGB (MediaPipe)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # Send the frame for asynchronous recognition
        frame_timestamp_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))
        recognizer.recognize_async(mp_image, frame_timestamp_ms)

        # Display the frame
        cv2.imshow('Hand Gesture Detection', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

Gesture: None (Confidence: 0.75)
Gesture: None (Confidence: 0.57)
Gesture: None (Confidence: 0.56)
Gesture: Pointing_Up (Confidence: 0.54)
Gesture: Pointing_Up (Confidence: 0.53)
Gesture: None (Confidence: 0.54)
Gesture: None (Confidence: 0.67)
Gesture: None (Confidence: 0.66)
Gesture: None (Confidence: 0.55)
Gesture: None (Confidence: 0.80)
Gesture: None (Confidence: 0.62)
Gesture: None (Confidence: 0.56)
Gesture: Pointing_Up (Confidence: 0.51)
Gesture: Pointing_Up (Confidence: 0.69)
Gesture: Pointing_Up (Confidence: 0.64)
Gesture: Pointing_Up (Confidence: 0.71)
Gesture: Pointing_Up (Confidence: 0.71)
Gesture: Pointing_Up (Confidence: 0.65)
Gesture: Pointing_Up (Confidence: 0.62)
Gesture: Pointing_Up (Confidence: 0.59)
Gesture: Pointing_Up (Confidence: 0.63)
Gesture: Pointing_Up (Confidence: 0.61)
Gesture: Pointing_Up (Confidence: 0.62)
Gesture: Pointing_Up (Confidence: 0.70)
Gesture: Pointing_Up (Confidence: 0.63)
Gesture: Pointing_Up (Confidence: 0.73)
Gesture: Pointing_Up (Confiden

In [4]:
import os
import cv2
import mediapipe as mp
from collections import deque
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ============================================================
# 1. MODEL PATH SETUP (change this if needed)
# ============================================================
model_path = "C:\\Users\\endop\\Downloads\\gesture_recognizer.task"   # <-- put file next to this script
# model_path = r'C:\full\path\to\gesture_recognizer.task'  # or absolute

if not os.path.isfile(model_path):
    raise FileNotFoundError(
        f"Model not found at: {os.path.abspath(model_path)}\n"
        "Download from: https://storage.googleapis.com/mediapipe-models/"
        "gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task"
    )
print(f"✅ Model found at: {os.path.abspath(model_path)}")

# ============================================================
# 2. TEMPORAL SMOOTHING BUFFER (module-level state)
# ============================================================
gesture_history = deque(maxlen=3)   # last 3 predictions
CONFIDENCE_THRESHOLD = 0.60

# ============================================================
# 3. RESULT CALLBACK (called by MediaPipe for every frame)
# ============================================================
def print_result(result, output_image, timestamp_ms):
    current = None
    score = 0.0
    if result.gestures and result.gestures[0]:
        top = result.gestures[0][0]
        score = top.score
        if top.score >= CONFIDENCE_THRESHOLD:
            current = top.category_name

    gesture_history.append(current)

    # Confirm only if the last 3 frames agree and are not None
    if (len(gesture_history) == 3
            and len(set(gesture_history)) == 1
            and gesture_history[0] is not None):
        print(f"✅ CONFIRMED: {gesture_history[0]}")
    elif current is not None:
        print(f"…  tentative: {current} ({score:.2f})")

# ============================================================
# 4. RECOGNIZER OPTIONS
# ============================================================
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.GestureRecognizerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.LIVE_STREAM,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    min_gesture_confidence=0.3,     # lowered so weak matches aren't silently dropped
    result_callback=print_result,
)

# ============================================================
# 5. MAIN LOOP
# ============================================================
with vision.GestureRecognizer.create_from_options(options) as recognizer:
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Cannot open webcam")

    frame_count = 0
    try:
        while cap.isOpened():
            ok, frame = cap.read()
            if not ok:
                break

            # MediaPipe needs RGB
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

            # Timestamp must be a strictly increasing integer
            recognizer.recognize_async(mp_image, frame_count)
            frame_count += 1

            cv2.imshow('Gesture Detection (press q to quit)', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()

✅ Model found at: C:\Users\endop\Downloads\gesture_recognizer.task


TypeError: GestureRecognizerOptions.__init__() got an unexpected keyword argument 'min_gesture_confidence'. Did you mean 'min_tracking_confidence'?

In [3]:
import cv2
import mediapipe as mp
import pyautogui
import time

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# --- Configure the HandLandmarker ---
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

landmarker = vision.HandLandmarker.create_from_options(options)

# --- CORRECT reference to the hand connections constant ---
HAND_CONNECTIONS = vision.HandLandmarksConnections.HAND_CONNECTIONS

def draw_landmarks_manual(frame, hand_landmarks):
    h, w, _ = frame.shape
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]
    
    for connection in HAND_CONNECTIONS:
        start_idx = connection.start
        end_idx = connection.end
        if start_idx < len(pts) and end_idx < len(pts):
            cv2.line(frame, pts[start_idx], pts[end_idx], (0, 255, 0), 2)
    
    for pt in pts:
        cv2.circle(frame, pt, 4, (0, 0, 255), -1)

def fingers_up(landmarks):
    fingers = []
    if landmarks[8].y < landmarks[6].y:
        fingers.append(1)
    else:
        fingers.append(0)
    if landmarks[12].y < landmarks[10].y:
        fingers.append(1)
    else:
        fingers.append(0)
    return fingers

cap = cv2.VideoCapture(0)
last_action_time = 0
delay = 1.0

while True:
    success, img = cap.read()
    if not success:
        break

    img = cv2.flip(img, 1)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    
    timestamp_ms = int(time.time() * 1000)
    result = landmarker.detect_for_video(mp_image, timestamp_ms)

    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:
            current_time = time.time()
            f = fingers_up(hand_landmarks)

            if f[0] == 1 and f[1] == 0:
                if current_time - last_action_time > delay:
                    pyautogui.press('right')
                    print("Next Slide (1 finger)")
                    last_action_time = current_time

            elif f[0] == 1 and f[1] == 1:
                if current_time - last_action_time > delay:
                    pyautogui.press('left')
                    print("Previous Slide (2 fingers)")
                    last_action_time = current_time

            draw_landmarks_manual(img, hand_landmarks)

    cv2.imshow("PowerPoint Gesture Control", img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
landmarker.close()

Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)
Previous Slide (2 fingers)


FailSafeException: PyAutoGUI fail-safe triggered from mouse moving to a corner of the screen. To disable this fail-safe, set pyautogui.FAILSAFE to False. DISABLING FAIL-SAFE IS NOT RECOMMENDED.